# Tome 9, Chapter 4: Localisation and Dimension (Python companion)

Hensel lifting and the Zassenhaus factoring algorithm, Newton's method in the local rings Z_(p), and how curves in Spec Z[x] meet the fibres over the primes.

*Before each solution, predict what the output will look like.*

## Exercises

**P4.1 (★★) CS application: Hensel lifting and factoring over $\Z$.** The Zassenhaus algorithm factors $f\in\Z[x]$ (square-free and monic, say) in three steps: factor $f$ modulo a prime $p$ (Berlekamp's algorithm, available in `sympy` as `gf_factor`); *lift* the factorisation to one modulo $p^k$ with $p^k$ larger than twice any coefficient of a possible factor, by Hensel's lemma (Newton's method in the local ring $\Z_{(p)}$, modulo powers of $p$); and *recombine*, trying products of the lifted factors with coefficients reduced to the symmetric range. (a) Implement Hensel lifting for $f\equiv gh\pmod p$ with $g,h$ coprime modulo $p$, and check that it doubles the precision at each step. (b) Factor $x^6-1$, $x^8+x^6-3x^4-3x^3+8x^2+2x-5$ (Knuth's example) and a product of random polynomials, and compare with `sympy.factor`. (c) The Swinnerton-Dyer polynomial $x^4-10x^2+1$ is irreducible over $\Z$ (it is the minimal polynomial of $\sqrt2+\sqrt3$, Exercise 4.2) but factors modulo every prime. Check this for the primes below $1000$, and explain why recombination is then needed.

**P4.2 (★) Square roots in $\Z_{(p)}$ and $p$-adic Newton.** An integer $a$ prime to $p$ (odd) that is a square modulo $p$ has square roots modulo every power $p^k$ (Hensel). Compute them with Newton's iteration $x\mapsto x-\frac{x^2-a}{2x}$ in $\Z/p^k$, find a square root of $-1$ modulo $5^{20}$ and of $2$ modulo $7^{20}$, and observe the number of correct base-$p$ digits double at each step. Why does the method fail for $p=2$ and $a=-1$, and what does $\operatorname{Spec}\Z[x]$ (Figure 4.1) have to say about it?

**P4.3 (★★) The fibres of a curve in $\operatorname{Spec}\Z[x]$.** For an irreducible $f\in\Z[x]$, the closed points of $V(f)$ above $p$ are the irreducible factors of $f$ modulo $p$ (Theorem 4.10). For $f=x^2+1$, $x^3-2$ and $x^4+x^3+x^2+x+1$, tabulate for the primes below $10^5$ the *factorisation types* (the degrees of the factors) and their frequencies. Compare with the predictions of the Chebotarev density theorem (Tome 12): the frequency of a type is the proportion of elements of the Galois group with that cycle structure ($1/2,1/2$ for $x^2+1$; $1/6$, $1/2$, $1/3$ for $x^3-2$, whose Galois group is $\mathfrak S_3$; $1/4,1/4,1/2$ for the cyclotomic quartic, whose group is $\Z/4$).

## Solutions

### P4.1

In [ ]:
import sympy as sp
from sympy.polys.galoistools import gf_factor_sqf, gf_gcdex
from sympy.polys.domains import ZZ
from itertools import combinations
import math, random
x = sp.Symbol("x")

def to_list(f):  return [int(c) for c in sp.Poly(f, x).all_coeffs()]
def sym(c, m):   c %= m; return c - m if c > m // 2 else c

def poly_mod(a, m):
    a = [c % m for c in a]
    while len(a) > 1 and a[0] == 0: a.pop(0)
    return a

def mul_mod(a, b, m):
    c = [0] * (len(a) + len(b) - 1)
    for i, u in enumerate(a):
        for j, v in enumerate(b):
            c[i + j] += u * v
    return poly_mod(c, m)

def divmod_mod(a, b, m):
    """Division of a by the monic b modulo m (works modulo any m because b is monic)."""
    a = poly_mod(a, m); q = [0] * max(len(a) - len(b) + 1, 1)
    a = a[:]
    while len(a) >= len(b) and any(a):
        c = a[0]; k = len(a) - len(b)
        q[len(q) - 1 - k] = c
        for i, bc in enumerate(b):
            a[i] = (a[i] - c * bc) % m
        a.pop(0)
    return poly_mod(q, m), poly_mod(a if a else [0], m)

def hensel_step(f, g, h, s, t, m):
    """Given f = g h mod m and s g + t h = 1 mod m, with h monic, deg s < deg h and deg t < deg g, return g*, h*, s*, t*
    with the same properties modulo m^2 (von zur Gathen and Gerhard, Algorithm 15.10)."""
    M = m * m
    e = poly_mod(sub(f, mul_mod(g, h, M)), M)
    q, r = divmod_mod(mul_mod(s, e, M), h, M)
    g2 = poly_mod(add(g, add(mul_mod(t, e, M), mul_mod(q, g, M))), M)
    h2 = poly_mod(add(h, r), M)
    b = poly_mod(sub(add(mul_mod(s, g2, M), mul_mod(t, h2, M)), [1]), M)
    c, d = divmod_mod(mul_mod(s, b, M), h2, M)
    s2 = poly_mod(sub(s, d), M)
    t2 = poly_mod(sub(t, add(mul_mod(t, b, M), mul_mod(c, g2, M))), M)
    return g2, h2, s2, t2

def add(a, b):
    n = max(len(a), len(b)); a = [0] * (n - len(a)) + a; b = [0] * (n - len(b)) + b
    return [u + v for u, v in zip(a, b)]
def sub(a, b):  return add(a, [-c for c in b])

A first test: $x^2+1\equiv(x-2)(x+2)\pmod5$. Lift to $5^2,5^4,5^8,\dots$ and watch the precision double: the lifted root of $x^2+1$ is a square root of $-1$ in $\Z/5^{2^k}$.

In [ ]:
f = to_list(x**2 + 1); p = 5
g, h = [1, -2 % p], [1, 2]
s_, t_, _ = gf_gcdex(g, h, p, ZZ)
s, t, m = [int(c) for c in s_], [int(c) for c in t_], p
for k in range(5):
    g, h, s, t = hensel_step(f, g, h, s, t, m)
    m = m * m
    root = (-g[1]) % m
    print(f"mod 5^{2**(k+1):<3d}: root {root},  root^2 + 1 = 5^{sp.multiplicity(5, root**2 + 1)} * (unit)")

Each step squares the modulus: the exponent of $5$ dividing $\text{root}^2+1$ doubles (at least), exactly like Newton's method doubling the number of correct digits. Now the whole algorithm, with multifactor lifting done by repeatedly splitting off one factor:

In [ ]:
def factor_zassenhaus(fexpr, p=None):
    f = to_list(fexpr)
    assert f[0] == 1, "monic only, for simplicity"
    n = len(f) - 1
    B = 2 * math.isqrt(n + 1) * 2**n * max(abs(c) for c in f) + 1   # Mignotte bound for factor coefficients (x2)
    if p is None:                     # a prime for which f stays square-free
        p = 3
        while sp.Poly(sp.gcd(sp.Poly(f, x, modulus=p), sp.Poly(f, x, modulus=p).diff(x))).degree() > 0 or f[0] % p == 0:
            p = sp.nextprime(p)
    _, facs = gf_factor_sqf([c % p for c in f], p, ZZ)
    facs = [[int(c) for c in fa] for fa in facs]
    k, m = 1, p
    while m <= B: m *= m; k *= 2
    # lift f = facs[0] * (rest) repeatedly
    lifted, rest, mm = [], f, None
    for i, g0 in enumerate(facs[:-1]):
        h0 = [1]
        for fa in facs[i + 1:]: h0 = poly_mod(mul_mod(h0, fa, p), p)
        s_, t_, _ = gf_gcdex(g0, h0, p, ZZ)
        g, h, s, t, mm = g0, h0, [int(c) for c in s_], [int(c) for c in t_], p
        while mm < m:
            g, h, s, t = hensel_step(rest, g, h, s, t, mm); mm *= mm
        lifted.append(g); rest = h
    lifted.append(rest)
    M = mm
    # recombination: try subsets of increasing size
    factors, remaining, F = [], list(range(len(lifted))), sp.Poly(f, x)
    size = 1
    while 2 * size <= len(remaining):
        for S in combinations(remaining, size):
            cand = [1]
            for i in S: cand = poly_mod(mul_mod(cand, lifted[i], M), M)
            cand = sp.Poly([sym(c, M) for c in cand], x)
            q, r = sp.div(F, cand)
            if r.is_zero:
                factors.append(cand.as_expr()); F = q
                remaining = [i for i in remaining if i not in S]
                break
        else:
            size += 1
    factors.append(F.as_expr())
    return p, len(lifted), factors

random.seed(4)
tests = [x**6 - 1, x**8 + x**6 - 3*x**4 - 3*x**3 + 8*x**2 + 2*x - 5,
         sp.expand((x**3 + 2*x - 7) * (x**4 - x + 3) * (x**2 + 5*x + 1))]
for fe in tests:
    p, r, facs = factor_zassenhaus(fe)
    ok = sp.expand(sp.prod(facs) - fe) == 0 and all(sp.Poly(fa, x).is_irreducible for fa in facs)
    print(f"p = {p}, {r} factors mod p ->", facs, " correct and irreducible:", ok)

Knuth's polynomial is irreducible over $\Z$ even though it splits into several factors modulo the chosen prime: no product of the lifted factors divides it. Now the Swinnerton-Dyer polynomial:

In [ ]:
sd = sp.Poly(x**4 - 10*x**2 + 1, x)
types = {}
for p in sp.primerange(3, 1000):
    if (2 * 3 * 96) % p == 0:          # the discriminant of x^4 - 10x^2 + 1 is 147456 = 2^14 3^2: skip p = 2, 3
        continue
    _, facs = gf_factor_sqf([c % p for c in sd.all_coeffs()], p, ZZ)
    t = tuple(sorted(len(fa) - 1 for fa in facs))
    types[t] = types.get(t, 0) + 1
print("factorisation types of x^4 - 10x^2 + 1 modulo the primes 3 < p < 1000 :", types)
print("irreducible over Z:", sd.is_irreducible)

It never stays irreducible modulo $p$: the types are $1+1+1+1$ and $2+2$, because its Galois group $\Z/2\times\Z/2$ has no element of order $4$. Modulo every prime there are at least two factors, so the algorithm must try products of factors; for products of $k$ such polynomials, $2^k$ combinations, the exponential worst case of Zassenhaus's method that the LLL algorithm (Tome C4) removed in 1982.

### P4.2

In [ ]:
def sqrt_mod_pk(a, p, k):
    m = p
    r = next(r for r in range(1, p) if (r * r - a) % p == 0)
    steps = []
    while m < p**k:
        m = min(m * m, p**k)
        r = (r - (r * r - a) * pow(2 * r, -1, m)) % m
        steps.append((m, r))
    return steps

for a, p in ((-1, 5), (2, 7)):
    for m, r in sqrt_mod_pk(a, p, 20):
        digits = sp.multiplicity(p, r * r - a) if r * r != a else float("inf")
        print(f"a = {a:2d}, modulus {p}^{int(round(math.log(m, p))):2d}: r = {r},  r^2 - a divisible by {p}^{digits}")
    print()

The exponent of $p$ dividing $r^2-a$ doubles at each step. For $p=2$, $a=-1$, the derivative $2x$ of $x^2+1$ vanishes modulo $2$ at the root $1$: Newton's method divides by $2x$, which is not a unit, and indeed $x^2\equiv-1\pmod4$ has no solution. In Figure 4.1 this is the point where the two branches of $V(x^2+1)$ meet the fibre at $2$ together: the prime $2$ *ramifies* in $\Z[i]$, $2=-i(1+i)^2$, and Hensel's lemma needs a simple root.

### P4.3

In [ ]:
from collections import Counter
from sympy.polys.galoistools import gf_factor
def fibre_types(fexpr, N):
    F = sp.Poly(fexpr, x)
    disc = int(sp.discriminant(F))
    cnt = Counter()
    for p in sp.primerange(2, N):
        if disc % p == 0:
            continue                       # ramified primes: finitely many, left out
        _, facs = gf_factor([c % p for c in F.all_coeffs()], p, ZZ)
        cnt[tuple(sorted((len(fa) - 1 for fa, e in facs), reverse=True))] += 1
    tot = sum(cnt.values())
    return {t: round(c / tot, 4) for t, c in sorted(cnt.items())}

for fe, pred in [(x**2 + 1, "(1,1): 1/2, (2,): 1/2"),
                 (x**3 - 2, "(1,1,1): 1/6, (2,1): 1/2, (3,): 1/3"),
                 (x**4 + x**3 + x**2 + x + 1, "(1,1,1,1): 1/4, (2,2): 1/4, (4,): 1/2")]:
    print(fe, " observed:", fibre_types(fe, 10**5), "  predicted:", pred)

The observed frequencies match the cycle types of the Galois groups to two or three decimals. For the cyclotomic polynomial the type is decided by $p$ modulo $5$: $p\equiv1$ gives four points, $p\equiv4$ two points of degree $2$, and $p\equiv2,3$ one point of degree $4$, so Dirichlet's theorem (Tome 5) already gives these frequencies. For $x^3-2$ no congruence condition decides the type, and Chebotarev's theorem is needed: the curve $V(x^3-2)$ crosses the fibres of $\operatorname{Spec}\Z[x]$ in a pattern governed by a non-abelian group.